# TP 15 — Méthodes d'ensemble

Données : © Open Food Facts contributors — ODbL

In [38]:
import joblib
import numpy as np
import optuna
import pandas as pd
import tempfile
import time

from IPython.display import Markdown, display
from nutriscope.ml.data import LABELS, column_types, load_dataset, split_xy
from nutriscope.ml.models import build_pipeline, tp14_models, boosting_models
from nutriscope.ml.protocol import CROSS_VALIDATION, SCORING, evaluate_cross_validation
from nutriscope.ml.tuning import time_one_trial, tune
from nutriscope.ml.errors import (
    build_results,
    confusion_by_row,
    error_gravity,
    performance_by_group,
    top_error_pairs,
)

from pathlib import Path
from sklearn.base import clone
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from xgboost import XGBClassifier

#### Partie 1
#### Chargement des données pour le TP

In [39]:
dataframe = load_dataset()
X_train, _, y_train, _ = split_xy(dataframe)
numeric_columns, categorical_columns = column_types(X_train)

print(X_train.shape)

(24076, 13)


#### Réentrainement dans les conditions imposées pour le TP

In [40]:
models = tp14_models()
names = [
    'logreg',
    'random_forest',
    'random_forest_balanced',
]

results_tp14 = [
    evaluate_cross_validation(
        build_pipeline(models[name], numeric_columns, categorical_columns),
        X_train,
        y_train,
        name,
    )
    for name in names
]

table_tp14 = pd.DataFrame(results_tp14).set_index('modele')
table_tp14.round(4)

,f1_macro_moy,f1_macro_std,accuracy_moy,accuracy_std,balanced_accuracy_moy,balanced_accuracy_std,fit_time_moy,score_time_moy
modele,,,,,,,,
logreg,0.7637,0.0080,0.8016,0.0055,0.7624,0.0075,1.0124,0.0357
random_forest,0.8551,0.0071,0.8751,0.0059,0.8508,0.0065,4.5009,0.1965
random_forest_balanced,0.8585,0.0047,0.8754,0.0041,0.8586,0.0043,4.7766,0.1863


#### Point de contrôle 1

In [41]:
# F1 macro (CV) lus dans le tableau du TP 14
REFERENCE_TP14 = {
    'logreg': 0.7605,
    'random_forest': 0.8530,
    'random_forest_balanced': 0.858,
}
THRESHOLD = 0.01

comparison = pd.DataFrame({
    'TP 14': pd.Series(REFERENCE_TP14),
    'TP 15': table_tp14.loc[list(REFERENCE_TP14), 'f1_macro_moy'],
})
comparison['écart'] = (comparison['TP 15'] - comparison['TP 14']).abs()
comparison['statut'] = comparison['écart'].apply(
    lambda gap: 'OK' if gap <= THRESHOLD else 'À VÉRIFIER'
)

display(
    comparison.style
    .format({
        'TP 14': '{:.4f}',
        'TP 15': '{:.4f}',
        'écart': '{:.4f}',
    })
    .map(
        lambda value: 'color: green' if value == 'OK' else 'color: red',
        subset=['statut'],
    )
)

assert (comparison['écart'] <= THRESHOLD).all(), (
    f'Écart supérieur à {THRESHOLD} : la préparation a bougé ?'
)

,TP 14,TP 15,écart,statut
logreg,0.7605,0.7637,0.0032,OK
random_forest,0.8530,0.8551,0.0021,OK
random_forest_balanced,0.8580,0.8585,0.0005,OK


In [42]:
display(Markdown(
    f"Les trois modèles du TP 14 sont retrouvés à {comparison['écart'].max():.4f} près au maximum "
    f"(seuil du sujet : {THRESHOLD}). L'écart reste inférieur à l'écart-type des plis : "
    f"il s'explique par le passage du split recalculé à la colonne `split` figée, "
    f"pas par un changement de préparation."
))

Les trois modèles du TP 14 sont retrouvés à 0.0032 près au maximum (seuil du sujet : 0.01). L'écart reste inférieur à l'écart-type des plis : il s'explique par le passage du split recalculé à la colonne `split` figée, pas par un changement de préparation.

#### Partie 2 - Le boosting

In [43]:
label_to_integer = {label: index for index, label in enumerate(LABELS)}
y_train_integer = y_train.map(label_to_integer)

results_boosting = [
    evaluate_cross_validation(
        build_pipeline(model, numeric_columns, categorical_columns),
        X_train,
        y_train_integer,
        name,
    )
    for name, model in boosting_models().items()
]

table_boosting = pd.DataFrame(results_boosting).set_index('modele')
table_boosting.round(4)

,f1_macro_moy,f1_macro_std,accuracy_moy,accuracy_std,balanced_accuracy_moy,balanced_accuracy_std,fit_time_moy,score_time_moy
modele,,,,,,,,
hist_gradient_boosting,0.8895,0.0038,0.9075,0.0037,0.8879,0.0036,5.3690,0.1684
xgboost,0.8946,0.0053,0.9112,0.0037,0.8932,0.0053,7.3443,0.1233
lightgbm,0.8945,0.0097,0.9116,0.0076,0.8937,0.0094,3.4871,0.6310


#### Budget annoncé avant le lancement

In [44]:
N_TRIALS = 30
trial_duration = time_one_trial(
    X_train, 
    y_train_integer, 
    numeric_columns, 
    categorical_columns
)

display(Markdown(
    f'**Budget annoncé avant le lancement** : {N_TRIALS} essais Optuna, 3 plis, MedianPruner. '
    f'Un essai dure environ {trial_duration:.0f} s, soit au moins {N_TRIALS * trial_duration / 60:.0f} min ; '
    f'on attend plutôt entre {N_TRIALS * trial_duration / 60:.0f} et {2 * N_TRIALS * trial_duration / 60:.0f} min. '
    f'Plafond fixé : 15 min.'
))

**Budget annoncé avant le lancement** : 30 essais Optuna, 3 plis, MedianPruner. Un essai dure environ 8 s, soit au moins 4 min ; on attend plutôt entre 4 et 8 min. Plafond fixé : 15 min.

#### Recherche et temps réel

In [45]:
start = time.perf_counter()
study = tune(X_train, y_train_integer, numeric_columns, categorical_columns, n_trials=N_TRIALS)
elapsed = time.perf_counter() - start
pruned = sum(trial.state == optuna.trial.TrialState.PRUNED for trial in study.trials)

display(Markdown(
    f'Recherche terminée en **{elapsed / 60:.1f} min** : {pruned} essais sur {N_TRIALS} abandonnés en cours de route. '
    f'Meilleur F1 macro sur 3 plis : {study.best_value:.4f}.'
))

Recherche terminée en **8.2 min** : 5 essais sur 30 abandonnés en cours de route. Meilleur F1 macro sur 3 plis : 0.8919.

#### Le réglage sert-il vraiment ?

In [46]:
tuned = XGBClassifier(**study.best_params, random_state=42, n_jobs=1)
result_tuned = evaluate_cross_validation(
    build_pipeline(tuned, numeric_columns, categorical_columns),
    X_train,
    y_train_integer,
    'xgboost_regle',
)

baseline = table_boosting.loc['xgboost']
gain = result_tuned['f1_macro_moy'] - baseline['f1_macro_moy']
threshold = 2 * baseline['f1_macro_std']
verdict = 'réel' if gain > threshold else 'non démontré : il reste dans le bruit des plis'

display(Markdown(
    f"XGBoost réglé : F1 macro {result_tuned['f1_macro_moy']:.4f} ± {result_tuned['f1_macro_std']:.4f}, "
    f"contre {baseline['f1_macro_moy']:.4f} avec les paramètres par défaut. "
    f'Gain : {gain:+.4f} (seuil de 2 écarts-types : {threshold:.4f}). **Le gain est {verdict}.**'
))

XGBoost réglé : F1 macro 0.8947 ± 0.0072, contre 0.8946 avec les paramètres par défaut. Gain : +0.0001 (seuil de 2 écarts-types : 0.0106). **Le gain est non démontré : il reste dans le bruit des plis.**

#### Etape 3 - Le tableau unique

| Condition | Où dans le code | Vérifié |
| --- | --- | --- |
| même préparation | `build_pipeline` appelle partout le même `preparation()` | ✅ |
| mêmes plis | un seul `CROSS_VALIDATION` (`StratifiedKFold(5, shuffle=True, random_state=42)`) dans `protocol.py` | ✅ |
| mêmes métriques | un seul `SCORING` dans `protocol.py` | ✅ |
| même `split` | colonne `split` figée du TP 13, lue par `split_xy`, jamais recalculée | ✅ |

In [47]:
_, X_test, _, y_test = split_xy(dataframe)
y_test_integer = y_test.map(label_to_integer)

tuned_model = XGBClassifier(**study.best_params, random_state=42, n_jobs=1)

models = {
    'logreg': tp14_models()['logreg'],
    'random_forest_balanced': tp14_models()['random_forest_balanced'],
    **boosting_models(),
    'xgboost_regle': tuned_model,
}

cv_table = pd.concat([
    table_tp14,
    table_boosting,
    pd.DataFrame([result_tuned]).set_index('modele'),
])

rows = []
for name, model in models.items():
    uses_integer = name not in ('logreg', 'random_forest_balanced')
    y_fit = y_train_integer if uses_integer else y_train
    y_true = y_test_integer if uses_integer else y_test

    pipeline = build_pipeline(model, numeric_columns, categorical_columns)

    start = time.perf_counter()
    pipeline.fit(X_train, y_fit)
    fit_time = time.perf_counter() - start

    start = time.perf_counter()
    predictions = pipeline.predict(X_test)
    predict_time = time.perf_counter() - start

    rows.append({
        'modele': name,
        'f1_macro_cv': cv_table.loc[name, 'f1_macro_moy'],
        'ecart_type_cv': cv_table.loc[name, 'f1_macro_std'],
        'f1_macro_test': f1_score(y_true, predictions, average='macro'),
        'accuracy_test': accuracy_score(y_true, predictions),
        'temps_ajustement_s': fit_time,
        'temps_prediction_s': predict_time,
    })

final_table = pd.DataFrame(rows).set_index('modele')
final_table['ecart_cv_test'] = (final_table['f1_macro_cv'] - final_table['f1_macro_test']).abs()
final_table.round(4)

,f1_macro_cv,ecart_type_cv,f1_macro_test,accuracy_test,temps_ajustement_s,temps_prediction_s,ecart_cv_test
modele,,,,,,,
logreg,0.7637,0.0080,0.7662,0.8028,1.0341,0.0206,0.0025
random_forest_balanced,0.8585,0.0047,0.8585,0.8746,4.6944,0.1845,0.0000
hist_gradient_boosting,0.8895,0.0038,0.8902,0.9075,2.3388,0.0601,0.0007
xgboost,0.8946,0.0053,0.8915,0.9075,6.8523,0.1211,0.0031
lightgbm,0.8945,0.0097,0.8911,0.9088,3.4317,0.5839,0.0033
xgboost_regle,0.8947,0.0072,0.8930,0.9096,15.1991,0.3200,0.0017


In [48]:
boosters = [
    'hist_gradient_boosting',
    'xgboost',
    'lightgbm',
]

default_row = final_table.loc['xgboost']
tuned_row = final_table.loc['xgboost_regle']
forest_f1 = final_table.loc['random_forest_balanced', 'f1_macro_test']

gain_min = final_table.loc[boosters, 'f1_macro_test'].min() - forest_f1
gain_max = final_table.loc[boosters, 'f1_macro_test'].max() - forest_f1

fit_ratio = tuned_row['temps_ajustement_s'] / default_row['temps_ajustement_s']
predict_ratio = tuned_row['temps_prediction_s'] / default_row['temps_prediction_s']

display(Markdown(
    f'- Les boostings dépassent la forêt équilibrée de {gain_min:.3f} à {gain_max:.3f} '
    f'de F1 macro sur le test.\n'
    f"- Entre boostings, les écarts restent du même ordre que l'écart-type des plis : "
    f'on ne les départage pas.\n'
    f'- Le modèle réglé est {fit_ratio:.1f} fois plus lent à ajuster '
    f'et {predict_ratio:.1f} fois plus lent à prédire que XGBoost par défaut, '
    f'pour un gain non démontré en validation croisée.'
))

- Les boostings dépassent la forêt équilibrée de 0.032 à 0.033 de F1 macro sur le test.
- Entre boostings, les écarts restent du même ordre que l'écart-type des plis : on ne les départage pas.
- Le modèle réglé est 2.2 fois plus lent à ajuster et 2.6 fois plus lent à prédire que XGBoost par défaut, pour un gain non démontré en validation croisée.

#### Etape 4 - Les confusions restantes

In [49]:
best_name = 'xgboost'
best_pipeline = build_pipeline(
    boosting_models()[best_name],
    numeric_columns,
    categorical_columns,
)
best_pipeline.fit(X_train, y_train_integer)

results = build_results(y_test_integer, best_pipeline.predict(X_test))
error_count = (results['reel'] != results['predit']).sum()

display(Markdown(
    f'Modèle analysé : **{best_name}**. '
    f'{error_count} erreurs sur {len(results)} produits du test ({error_count / len(results):.1%}).'
))

Modèle analysé : **xgboost**. 552 erreurs sur 5965 produits du test (9.3%).

In [50]:
display(Markdown('**Matrice de confusion, normalisée par ligne**'))
display(confusion_by_row(results, LABELS).style.format('{:.1%}').background_gradient(cmap='Blues', axis=None))

display(Markdown('**Les cinq couples (réel, prédit) les plus fréquents**'))
display(top_error_pairs(results, LABELS))

display(Markdown('**Gravité des erreurs**'))
display(error_gravity(results))

display(Markdown('**Performance par rayon, du pire au meilleur**'))
display(performance_by_group(results, dataframe.loc[X_test.index, 'pnns_groups_1']).round(3))

**Matrice de confusion, normalisée par ligne**

prédit,a,b,c,d,e
réel,,,,,
a,91.2%,6.4%,1.6%,0.3%,0.5%
b,9.9%,79.0%,11.0%,0.2%,0.0%
c,1.2%,5.8%,87.5%,5.4%,0.1%
d,0.1%,0.4%,3.1%,93.4%,3.1%
e,0.0%,0.1%,0.4%,4.9%,94.6%


**Les cinq couples (réel, prédit) les plus fréquents**

,réel,prédit,effectif,part des erreurs
0,e,d,85,0.154
1,c,b,70,0.127
2,b,c,69,0.125
3,c,d,66,0.120
4,b,a,62,0.112


**Gravité des erreurs**

,effectif,part des erreurs
ecart,,
un cran,501,0.908
deux crans,43,0.078
trois crans ou plus,8,0.014


**Performance par rayon, du pire au meilleur**

,effectif,accuracy,f1_macro
rayon,,,
Alcoholic beverages,10,0.600,0.390
Cereals and potatoes,603,0.847,0.819
Beverages,471,0.866,0.850
Fruits and vegetables,403,0.871,0.837
Composite foods,494,0.893,0.863
Salty snacks,388,0.897,0.891
Fat and sauces,422,0.905,0.896
Fish Meat Eggs,1110,0.919,0.887
Milk and dairy products,715,0.930,0.901


In [51]:
probabilities = best_pipeline.predict_proba(X_test)
sorted_probabilities = np.sort(probabilities, axis=1)
results['marge'] = sorted_probabilities[:, -1] - sorted_probabilities[:, -2]

errors = results[results['reel'] != results['predit']]

between_b_c = errors[errors['reel'].isin([1, 2]) & errors['predit'].isin([1, 2])]
between_d_e = errors[errors['reel'].isin([3, 4]) & errors['predit'].isin([3, 4])]

selection = pd.concat([
    between_b_c.nsmallest(3, 'marge'),
    between_d_e.nsmallest(2, 'marge'),
])

summary_view = selection[['reel', 'predit', 'marge']].copy()
summary_view['reel'] = summary_view['reel'].map(lambda index: LABELS[index])
summary_view['predit'] = summary_view['predit'].map(lambda index: LABELS[index])

display(summary_view.round(3))
display(dataframe.loc[selection.index].T)

,reel,predit,marge
23730,b,c,0.006
27307,b,c,0.010
24154,b,c,0.012
22857,e,d,0.001
6625,e,d,0.006


,23730,27307,24154,22857,6625
code,5054268892285,8010706700120,5400101255371,4104420007987,3113010133037
product_name,Easy Oats Original,Double Concentre De Tomate,Les Belges Stoemp aux Carottes,Mini Schoko Lebkuchen,Foie Gras de Canard Entier au Cognac
brands,Tesco,Giaguaro,"Carrefour,Les Belges","Alnatura,Alnatura GmbH",Larnaudie
pnns_groups_1,Cereals and potatoes,Fruits and vegetables,Composite foods,Sugary snacks,Salty snacks
pnns_groups_2,Breakfast cereals,Vegetables,One-dish meals,Biscuits and cakes,Salty and fatty products
nova_group,1,3,4,4,3
additives_n,0,1,1,2,2
energy-kcal_100g,92.270531,74.0,80.0,420.0,571.0
fat_100g,2.6,0.5,2.7,17.0,59.7
saturated-fat_100g,1.2,0.01,1.8,6.7,26.7


| Produit | Diagnostic | Justification |
| --- | --- | --- |
| Easy Oats Original | donnée fausse (probable) | valeurs « as sold » à −72 % en énergie et −85 % en fibres vs la catégorie : profil d'un produit préparé |
| Double Concentré de Tomate | limite du modèle | score 1, effets contraires fruits-légumes et sel, proche de la frontière b/c |
| Stoemp aux Carottes | limite du modèle | fruits-légumes estimé à 40,24 %, juste au-dessus du seuil de 40 % |
| Mini Schoko Lebkuchen | limite du modèle | score 20, à un point du seuil e |
| Foie Gras de Canard | limite du modèle | score 19, pile sur le seuil e ; fibres absentes de la fiche, imputées par la médiane |

#### Etape 5 - L'ADR

In [52]:
candidates = [
    'xgboost',
    'xgboost_regle',
    'hist_gradient_boosting',
    'random_forest_balanced',
]

sizes = {}
with tempfile.TemporaryDirectory() as folder:
    for name in candidates:
        uses_integer = name != 'random_forest_balanced'
        pipeline = build_pipeline(clone(models[name]), numeric_columns, categorical_columns)
        pipeline.fit(X_train, y_train_integer if uses_integer else y_train)

        path = Path(folder) / f'{name}.joblib'
        joblib.dump(pipeline, path, compress=3)
        sizes[name] = path.stat().st_size / 1e6

adr_table = final_table.loc[
    candidates,
    ['f1_macro_cv', 'ecart_type_cv', 'temps_ajustement_s'],
].copy()
adr_table['taille_Mo'] = pd.Series(sizes)
adr_table['latence_ms_par_produit'] = (
    final_table.loc[candidates, 'temps_prediction_s'] / len(X_test) * 1000
)
adr_table.round(4)

,f1_macro_cv,ecart_type_cv,temps_ajustement_s,taille_Mo,latence_ms_par_produit
modele,,,,,
xgboost,0.8946,0.0053,6.8523,1.8394,0.0203
xgboost_regle,0.8947,0.0072,15.1991,4.4583,0.0536
hist_gradient_boosting,0.8895,0.0038,2.3388,0.8434,0.0101
random_forest_balanced,0.8585,0.0047,4.6944,28.7279,0.0309


In [54]:
chosen = adr_table.loc['xgboost']
tuned = adr_table.loc['xgboost_regle']
hgb = adr_table.loc['hist_gradient_boosting']
forest = adr_table.loc['random_forest_balanced']

fit_ratio = tuned['temps_ajustement_s'] / chosen['temps_ajustement_s']
revision_threshold = chosen['f1_macro_cv'] - 2 * chosen['ecart_type_cv']

tuned_gain = tuned['f1_macro_cv'] - chosen['f1_macro_cv']
tuned_verdict = (
    'gain démontré' if tuned_gain > 2 * chosen['ecart_type_cv']
    else 'gain non démontré (inférieur à 2 écarts-types)'
)

hgb_gap = chosen['f1_macro_cv'] - hgb['f1_macro_cv']
hgb_verdict = (
    'non distinguable de A' if hgb_gap <= chosen['ecart_type_cv']
    else f'inférieur à A de {hgb_gap:.4f}'
)

forest_gap = chosen['f1_macro_cv'] - forest['f1_macro_cv']


def metric(values):
    return f"{values['f1_macro_cv']:.4f} ± {values['ecart_type_cv']:.4f}"


def cost(values):
    return (
        f"{values['taille_Mo']:.1f} Mo / {values['latence_ms_par_produit']:.3f} ms par produit / "
        f"réentraînement {values['temps_ajustement_s']:.1f} s"
    )


adr = f"""# ADR-001 : Nous retenons XGBoost avec ses paramètres par défaut pour prédire le Nutri-Score

## Contexte

Prédire la classe Nutri-Score (a à e) d'un produit Open Food Facts à partir de ses
nutriments et de quelques variables dérivées. Les modèles sont comparés dans les mêmes
conditions : même préparation, mêmes 5 plis, mêmes métriques, même split figé. Aucune
variable issue du score officiel n'est utilisée comme entrée.

## Options considérées

| Option | Métrique clé (F1 macro CV) | Coût (taille / latence / réentraînement) | Autre critère |
| --- | --- | --- | --- |
| A. XGBoost, paramètres par défaut | {metric(chosen)} | {cost(chosen)} ; dépendance `xgboost`, cible entière | importances natives |
| B. XGBoost réglé (Optuna, 30 essais) | {metric(tuned)} | {cost(tuned)} ; ajoute Optuna | {tuned_verdict}, réentraînement {fit_ratio:.1f} fois plus long |
| C. HistGradientBoosting | {metric(hgb)} | {cost(hgb)} ; scikit-learn seul | {hgb_verdict} ; importance par permutation seulement |
| D. Forêt aléatoire équilibrée | {metric(forest)} | {cost(forest)} ; scikit-learn seul | inférieure à A de {forest_gap:.3f} ; importances natives |

## Décision

Nous retenons XGBoost avec ses paramètres par défaut (option A).

## Conséquences

- Ce que ce choix facilite : un modèle léger ({chosen['taille_Mo']:.1f} Mo), rapide à
  servir ({chosen['latence_ms_par_produit']:.3f} ms par produit) et à réentraîner
  ({chosen['temps_ajustement_s']:.1f} s).
- Ce qu'on sacrifie : le réglage des hyperparamètres (gain non démontré aujourd'hui) et
  l'indépendance vis-à-vis de la dépendance `xgboost`, qu'évitent les options C et D.

## Ce qui déclencherait une révision

- Le F1 macro en validation croisée sur de nouvelles données passe sous {revision_threshold:.4f}.
- Un réglage futur dépasse deux écarts-types de gain en validation croisée.
- La taille ou la latence dépasse la contrainte du service qui utilisera le modèle.
"""

print(adr)

# ADR-001 : Nous retenons XGBoost avec ses paramètres par défaut pour prédire le Nutri-Score

## Contexte

Prédire la classe Nutri-Score (a à e) d'un produit Open Food Facts à partir de ses
nutriments et de quelques variables dérivées. Les modèles sont comparés dans les mêmes
conditions : même préparation, mêmes 5 plis, mêmes métriques, même split figé. Aucune
variable issue du score officiel n'est utilisée comme entrée.

## Options considérées

| Option | Métrique clé (F1 macro CV) | Coût (taille / latence / réentraînement) | Autre critère |
| --- | --- | --- | --- |
| A. XGBoost, paramètres par défaut | 0.8946 ± 0.0053 | 1.8 Mo / 0.020 ms par produit / réentraînement 6.9 s ; dépendance `xgboost`, cible entière | importances natives |
| B. XGBoost réglé (Optuna, 30 essais) | 0.8947 ± 0.0072 | 4.5 Mo / 0.054 ms par produit / réentraînement 15.2 s ; ajoute Optuna | gain non démontré (inférieur à 2 écarts-types), réentraînement 2.2 fois plus long |
| C. HistGradientBoosting | 0.8895 ± 0.